
## Folder 04 — File 03: Silver Orders:


**Transformation rules:**

- Trim order_id, customer_id, currency, and sales_channel.
- Convert order_timestamp into a timestamp.
- Standardize order_status to uppercase.
- Convert order_total to DECIMAL(12,2).
- Quarantine missing IDs, invalid timestamps, unsupported statuses, missing currency, and negative totals.
- Deduplicate using order_id.
- Save valid records to silver_orders.


Allowed statuses in this dataset: **DELIVERED, SHIPPED, PROCESSING, CANCELLED**.

In [0]:

## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"

BRONZE_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_orders"
)

SILVER_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_orders"
)

QUARANTINE_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.quarantine_orders"
)

ALLOWED_ORDER_STATUSES = [
    "DELIVERED",
    "SHIPPED",
    "PROCESSING",
    "CANCELLED"
]


## 3. Create Silver schema

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{CATALOG}`.`{SILVER_SCHEMA}`"
)

print("Silver schema is ready.")


## 4. Read Bronze orders

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise Exception(
        f"Bronze table not found: {BRONZE_TABLE}. "
        "Run Folder 03 first."
    )

bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze order records: {bronze_df.count()}")

display(bronze_df.limit(10))


## 5. Clean and standardize order columns

cleaned_df = (
    bronze_df

    .withColumn(
        "order_id",
        F.trim(F.col("order_id").cast("string"))
    )
    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id").cast("string"))
    )
    .withColumn(
        "order_status",
        F.upper(F.trim(F.col("order_status").cast("string")))
    )
    .withColumn(
        "currency",
        F.upper(F.trim(F.col("currency").cast("string")))
    )
    .withColumn(
        "sales_channel",
        F.initcap(F.trim(F.col("sales_channel").cast("string")))
    )

    # Convert order timestamp

    .withColumn(
        "order_timestamp",
        F.expr(
            "try_cast(order_timestamp AS TIMESTAMP)"
        )
    )

    # Convert order total

    .withColumn(
        "order_total",
        F.expr(
            "try_cast(order_total AS DECIMAL(12,2))"
        )
    )
)

print("Order standardization completed.")

display(cleaned_df.limit(10))


## 6. Identify invalid order records

invalid_condition = (
    F.col("order_id").isNull()
    | (F.col("order_id") == "")

    | F.col("customer_id").isNull()
    | (F.col("customer_id") == "")

    | F.col("order_timestamp").isNull()

    | F.col("order_status").isNull()
    | (~F.col("order_status").isin(ALLOWED_ORDER_STATUSES))

    | F.col("currency").isNull()
    | (F.col("currency") == "")

    | F.col("order_total").isNull()
    | (F.col("order_total") < 0)
)

invalid_df = (
    cleaned_df
    .filter(invalid_condition)
    .withColumn(
        "_quarantine_reason",
        F.concat_ws(
            "; ",

            F.when(
                F.col("order_id").isNull()
                | (F.col("order_id") == ""),
                F.lit("Missing or empty order_id")
            ),

            F.when(
                F.col("customer_id").isNull()
                | (F.col("customer_id") == ""),
                F.lit("Missing or empty customer_id")
            ),

            F.when(
                F.col("order_timestamp").isNull(),
                F.lit("Missing or invalid order_timestamp")
            ),

            F.when(
                F.col("order_status").isNull()
                | (~F.col("order_status").isin(ALLOWED_ORDER_STATUSES)),
                F.lit("Invalid order_status")
            ),

            F.when(
                F.col("currency").isNull()
                | (F.col("currency") == ""),
                F.lit("Missing currency")
            ),

            F.when(
                F.col("order_total").isNull(),
                F.lit("Missing or invalid order_total")
            ),

            F.when(
                F.col("order_total") < 0,
                F.lit("Negative order_total")
            )
        )
    )
)

valid_df = cleaned_df.filter(~invalid_condition)

print(f"Valid order records: {valid_df.count()}")
print(f"Invalid order records: {invalid_df.count()}")


## 7. Remove duplicate order records

deduplication_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("_ingestion_timestamp").desc_nulls_last(),
        F.col("_source_file").desc_nulls_last()
    )
)

deduplicated_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(deduplication_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

print(
    f"Records after deduplication: "
    f"{deduplicated_df.count()}"
)


## 8. Save invalid records to quarantine

(
    invalid_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print(f"Quarantine table created: {QUARANTINE_TABLE}")


## 9. Save clean records to Silver Delta table

(
    deduplicated_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver table created: {SILVER_TABLE}")


## 10. Validate Silver order records

silver_df = spark.table(SILVER_TABLE)

total_silver_records = silver_df.count()

duplicate_order_groups = (
    silver_df
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_order_records = (
    silver_df
    .filter(
        F.col("order_id").isNull()
        | F.col("customer_id").isNull()
        | F.col("order_timestamp").isNull()
        | F.col("order_status").isNull()
        | (~F.col("order_status").isin(ALLOWED_ORDER_STATUSES))
        | F.col("order_total").isNull()
        | (F.col("order_total") < 0)
    )
    .count()
)

print("=" * 50)
print("SILVER ORDER VALIDATION")
print("=" * 50)

print(f"Silver records: {total_silver_records}")
print(f"Duplicate order groups: {duplicate_order_groups}")
print(f"Invalid order records: {invalid_order_records}")

if (
    duplicate_order_groups > 0
    or invalid_order_records > 0
):
    raise Exception("Silver order validation failed.")

print("Silver order validation passed.")


## 11. Analyze order statuses

display(
    silver_df
    .groupBy("order_status")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("order_total").alias("total_order_value")
    )
    .orderBy("order_status")
)


## 12. Display final Silver order data

display(
    silver_df
    .orderBy("order_timestamp")
    .limit(20)
)

## 13. Final summary

print("=" * 50)
print("ORDER SILVER TRANSFORMATION COMPLETED")
print("=" * 50)

print(f"Bronze table     : {BRONZE_TABLE}")
print(f"Silver table     : {SILVER_TABLE}")
print(f"Quarantine table : {QUARANTINE_TABLE}")

print(f"Bronze records   : {bronze_df.count()}")
print(f"Silver records   : {total_silver_records}")
print(f"Invalid records  : {invalid_df.count()}")

print("=" * 50)
